# 真實 TCCIP 網格資料：GP Kernel AIC 與 Spatial CV Workflow

針對真實 TCCIP grid data 上已估好的 GEV 參數，建立一個比較 RBF 與 Matérn kernel 的流程。

研究問題是：

> 在真實資料沒有 true parameter surface 的情況下，如何判斷要用 RBF 還是 Matérn 來建立 GEV 參數曲面？

這裡不只用 visual variogram ，而是結合三個角度判斷量化

1. Empirical variogram：看空間結構與平滑程度。
2. GP log marginal likelihood / AIC：量化資料比較支持哪個 kernel。
3. Spatial cross-validation：檢查 kernel 對未見區域的預測能力。


## 1. 方法流程

本研究在真實 grid data 上採用以下流程：

```text
每個 grid point 先估 GEV 參數
        ↓
得到 grid-level mu_hat, sigma_hat, xi_hat
        ↓
對每個參數分別 fit Gaussian Process
        ↓
比較 RBF 與 Matérn 的 log marginal likelihood / AIC
        ↓
用 spatial CV 檢查預測能力
        ↓
決定真實資料下哪個 kernel 較合理
```

注意：這裡雖然資料本身是 grid，但經過缺值篩選後是 1412 個有效格點，因此仍可視為空間點資料來 fit GP。


## 2. AIC 的判斷邏輯

Gaussian Process 會根據 kernel 產生 covariance matrix，並計算資料在該 kernel 假設下的 log marginal likelihood。

AIC 定義為：

$$
AIC = 2k - 2\log L
$$

其中：

- $k$：模型中被估計的 kernel hyperparameters 數量
- $\log L$：GP 的 log marginal likelihood
- AIC 越小，代表在考慮模型複雜度後，資料越支持該 kernel

因此可以用：

```text
RBF AIC < Matérn AIC     -> 資料較支持 RBF
Matérn AIC < RBF AIC     -> 資料較支持 Matérn
```


In [1]:
from pathlib import Path
import warnings

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from shapely.geometry import Point
from sklearn.cluster import KMeans
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel as C, Matern, RBF, WhiteKernel
from sklearn.metrics import mean_absolute_error, mean_squared_error, silhouette_score

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FIG_DIR = PROJECT_ROOT / "results" / "figures"
TABLE_DIR = PROJECT_ROOT / "results" / "tables"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH = PROCESSED_DIR / "grid_station_gev_params_with_loc.csv"
print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_PATH exists:", DATA_PATH.exists())


PROJECT_ROOT: c:\Users\User.DESKTOP-4RV84M1\Desktop\論文\fast parameter estimate\fast_parameter_using_NN\experiments\window_data
DATA_PATH exists: True


In [2]:
df = pd.read_csv(DATA_PATH)
df = df[["station", "lon", "lat", "mu_hat", "sigma_hat", "log_sigma_hat", "xi_hat"]].dropna().copy()

print("n grid points:", len(df))
print("lon range:", df["lon"].min(), df["lon"].max())
print("lat range:", df["lat"].min(), df["lat"].max())
df.head()


n grid points: 1412
lon range: 119.45 122.1
lat range: 21.9 25.65


,station,lon,lat,mu_hat,sigma_hat,log_sigma_hat,xi_hat
0,G119.45_23.20,119.45,23.20,31.698334,0.238125,-1.434958,0.227292
1,G119.50_23.35,119.50,23.35,31.679490,0.279867,-1.273440,0.180436
2,G119.50_23.40,119.50,23.40,31.653676,0.273900,-1.294992,0.181252
3,G119.50_23.50,119.50,23.50,31.540771,0.204351,-1.587916,0.236057
4,G119.50_23.60,119.50,23.60,31.527162,0.315273,-1.154317,0.129411


## 3. 建立 RBF 與 Matérn GP

這裡比較的 kernel 為：

$$
K = C 	imes K_{spatial} + K_{noise}
$$

其中 $K_{spatial}$ 分別使用 RBF 或 Matérn。

Matérn 另外比較三個常見平滑度：

- $
u=0.5$：較粗糙
- $
u=1.5$：中等平滑
- $
u=2.5$：較平滑

為了避免真實 grid 點數太多導致 exact GP 運算過慢，AIC 計算固定抽樣最多 800 個 grid points。這與原本 Kriging 流程中 `max_train=800` 的設定一致。


In [3]:
RANDOM_STATE = 111
MAX_TRAIN = 800
TARGETS = {
    "mu": "mu_hat",
    "log_sigma": "log_sigma_hat",
    "xi": "xi_hat",
}


def standardize_coords(coords):
    coords = np.asarray(coords, dtype=np.float64)
    mean = coords.mean(axis=0)
    std = coords.std(axis=0)
    std[std == 0] = 1.0
    return (coords - mean) / std, mean, std


def make_kernel(kernel_name, nu=None):
    if kernel_name == "RBF":
        spatial = RBF(length_scale=1.0, length_scale_bounds=(1e-2, 10.0))
    elif kernel_name == "Matern":
        spatial = Matern(length_scale=1.0, length_scale_bounds=(1e-2, 10.0), nu=nu)
    else:
        raise ValueError(kernel_name)

    return C(1.0, (1e-2, 1e2)) * spatial + WhiteKernel(
        noise_level=1e-4,
        noise_level_bounds=(1e-8, 1e-1),
    )


def sample_for_gp(data, max_train=MAX_TRAIN, random_state=RANDOM_STATE):
    if len(data) <= max_train:
        return data.copy()
    return data.sample(max_train, random_state=random_state).copy()


## 4. Log Marginal Likelihood 與 AIC

這一段會對每個 GEV 參數分別 fit GP，並輸出：

- log marginal likelihood
- AIC
- optimizer 找到的 kernel hyperparameters

AIC 越小代表越好。


In [4]:
kernel_specs = [
    {"kernel": "RBF", "nu": np.nan},
    {"kernel": "Matern", "nu": 0.5},
    {"kernel": "Matern", "nu": 1.5},
    {"kernel": "Matern", "nu": 2.5},
]

rows = []
for target_name, target_col in TARGETS.items():
    fit_df = sample_for_gp(df[["lon", "lat", target_col]].dropna())
    X, _, _ = standardize_coords(fit_df[["lon", "lat"]].to_numpy())
    y = fit_df[target_col].to_numpy(dtype=np.float64)

    for spec in kernel_specs:
        gp = GaussianProcessRegressor(
            kernel=make_kernel(spec["kernel"], None if pd.isna(spec["nu"]) else float(spec["nu"])),
            n_restarts_optimizer=5,
            normalize_y=True,
            random_state=RANDOM_STATE,
        )
        gp.fit(X, y)
        log_likelihood = float(gp.log_marginal_likelihood(gp.kernel_.theta))
        k = int(len(gp.kernel_.theta))
        aic = 2 * k - 2 * log_likelihood
        rows.append({
            "target": target_name,
            "kernel": spec["kernel"],
            "nu": spec["nu"],
            "n_train": len(fit_df),
            "n_hyperparameters": k,
            "log_marginal_likelihood": log_likelihood,
            "AIC": float(aic),
            "fitted_kernel": str(gp.kernel_),
        })

aic_results = pd.DataFrame(rows).sort_values(["target", "AIC"]).reset_index(drop=True)
aic_results.to_csv(TABLE_DIR / "real_grid_gp_aic_kernel_selection.csv", index=False, encoding="utf-8-sig")
aic_results


,target,kernel,nu,n_train,n_hyperparameters,log_marginal_likelihood,AIC,fitted_kernel
0,log_sigma,Matern,0.5,800,3,-836.446988,1678.893977,"0.946**2 * Matern(length_scale=0.237, nu=0.5) ..."
1,log_sigma,Matern,1.5,800,3,-850.883801,1707.767603,"0.892**2 * Matern(length_scale=0.136, nu=1.5) ..."
2,log_sigma,Matern,2.5,800,3,-863.301308,1732.602616,"0.879**2 * Matern(length_scale=0.117, nu=2.5) ..."
3,log_sigma,RBF,NaN,800,3,-895.190946,1796.381891,0.863**2 * RBF(length_scale=0.0929) + WhiteKer...
4,mu,Matern,0.5,800,3,-243.261444,492.522888,"0.999**2 * Matern(length_scale=1.17, nu=0.5) +..."
5,mu,Matern,1.5,800,3,-259.528817,525.057633,"0.794**2 * Matern(length_scale=0.413, nu=1.5) ..."
6,mu,Matern,2.5,800,3,-272.802183,551.604366,"0.765**2 * Matern(length_scale=0.385, nu=2.5) ..."
7,mu,RBF,NaN,800,3,-292.152910,590.305819,0.746**2 * RBF(length_scale=0.338) + WhiteKern...
8,xi,Matern,0.5,800,3,-887.275943,1780.551887,"1.01**2 * Matern(length_scale=0.215, nu=0.5) +..."
9,xi,Matern,1.5,800,3,-898.993613,1803.987226,"0.942**2 * Matern(length_scale=0.128, nu=1.5) ..."


In [5]:
best_aic = aic_results.loc[aic_results.groupby("target")["AIC"].idxmin()].sort_values("target")
best_aic[["target", "kernel", "nu", "AIC", "log_marginal_likelihood", "fitted_kernel"]]


,target,kernel,nu,AIC,log_marginal_likelihood,fitted_kernel
0,log_sigma,Matern,0.5,1678.893977,-836.446988,"0.946**2 * Matern(length_scale=0.237, nu=0.5) ..."
4,mu,Matern,0.5,492.522888,-243.261444,"0.999**2 * Matern(length_scale=1.17, nu=0.5) +..."
8,xi,Matern,0.5,1780.551887,-887.275943,"1.01**2 * Matern(length_scale=0.215, nu=0.5) +..."


## 5. Spatial Fold Selection: Elbow and Silhouette

原本 spatial CV 先使用 $K=5$ 個 KMeans spatial folds，但 $K$ 不應該完全固定。

這裡先比較不同 $K$ 的空間分群品質：

- **Elbow method**：看 within-cluster sum of squares 是否開始趨緩。
- **Silhouette score**：看空間群是否分得清楚。
- **Fold-size balance**：避免某些 testing fold 太小，導致 spatial CV 不穩。

注意：elbow 與 silhouette 只是在檢查「空間分群是否合理」，最後仍要搭配 spatial CV RMSE 判斷 kernel 預測能力。

![KMeans elbow and silhouette](../results/figures/real_grid_kmeans_elbow_silhouette.png)


In [ ]:
K_CANDIDATES = range(2, 13)

coords_for_k = df[["lon", "lat"]].to_numpy(dtype=np.float64)
coords_for_k_std, _, _ = standardize_coords(coords_for_k)

k_rows = []
for k in K_CANDIDATES:
    kmeans = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=30)
    labels = kmeans.fit_predict(coords_for_k_std)
    counts = pd.Series(labels).value_counts().sort_index()

    k_rows.append({
        "K": k,
        "inertia": float(kmeans.inertia_),
        "silhouette": float(silhouette_score(coords_for_k_std, labels)),
        "min_fold_size": int(counts.min()),
        "max_fold_size": int(counts.max()),
        "fold_size_ratio": float(counts.max() / counts.min()),
    })

k_selection = pd.DataFrame(k_rows)
k_selection.to_csv(TABLE_DIR / "real_grid_kmeans_fold_selection.csv", index=False, encoding="utf-8-sig")

fig, axes = plt.subplots(1, 2, figsize=(10, 4.2), dpi=160)

axes[0].plot(k_selection["K"], k_selection["inertia"], marker="o", linewidth=2)
axes[0].axvline(5, color="crimson", linestyle="--", linewidth=1.5, label="current K=5")
axes[0].set_title("Elbow method")
axes[0].set_xlabel("Number of spatial folds K")
axes[0].set_ylabel("Within-cluster sum of squares")
axes[0].legend()
axes[0].grid(alpha=0.25)

axes[1].plot(k_selection["K"], k_selection["silhouette"], marker="o", linewidth=2, color="darkgreen")
axes[1].axvline(5, color="crimson", linestyle="--", linewidth=1.5, label="current K=5")
axes[1].set_title("Silhouette score")
axes[1].set_xlabel("Number of spatial folds K")
axes[1].set_ylabel("Silhouette score")
axes[1].legend()
axes[1].grid(alpha=0.25)

fig.suptitle("KMeans spatial fold selection for real TCCIP grid data")
fig.tight_layout()
fig.savefig(FIG_DIR / "real_grid_kmeans_elbow_silhouette.png", bbox_inches="tight")
plt.show()

k_selection


## 6. Selected Number of Spatial Folds

根據目前資料：

- $K=2$ 的 silhouette 最高，但 fold 太少，spatial CV 較不嚴格。
- $K=5$ 的 fold size 很平均，且 elbow 曲線已經開始趨緩。
- $K \geq 8$ 時，最小 fold size 變得很小，testing set 不穩定。

因此目前先保留：

$$
K = 5
$$

後續若要更完整，可以把 $K=3,4,5,6,7$ 都跑 spatial CV，檢查 kernel 排名是否穩定。


In [ ]:
SELECTED_K = 5
N_FOLDS = SELECTED_K
print("Selected spatial folds K =", N_FOLDS)


## 7. Spatial Cross-Validation

AIC 是看資料比較支持哪個 kernel；spatial CV 則是看預測能力。

這裡用經緯度做 KMeans 空間分群，每次留一個空間群當 testing set，其餘群當 training set。

這比 random CV 更合理，因為空間資料有相關性；如果隨機切資料，testing grid point 旁邊可能剛好有 training grid point，結果會過度樂觀。


In [6]:
# N_FOLDS is selected in the previous section.
coords_for_group = df[["lon", "lat"]].to_numpy(dtype=np.float64)
coords_std, _, _ = standardize_coords(coords_for_group)

df_cv = df.copy()
df_cv["spatial_fold"] = KMeans(n_clusters=N_FOLDS, random_state=RANDOM_STATE, n_init=20).fit_predict(coords_std)

df_cv["spatial_fold"].value_counts().sort_index()


spatial_fold
0    267
1    269
2    277
3    318
4    281
Name: count, dtype: int64

In [7]:
def fit_predict_gp(train_df, test_df, target_col, kernel_name, nu=None, max_train=MAX_TRAIN):
    if len(train_df) > max_train:
        train_df = train_df.sample(max_train, random_state=RANDOM_STATE).copy()

    X_train_raw = train_df[["lon", "lat"]].to_numpy(dtype=np.float64)
    X_train, mean, std = standardize_coords(X_train_raw)
    X_test = (test_df[["lon", "lat"]].to_numpy(dtype=np.float64) - mean) / std
    y_train = train_df[target_col].to_numpy(dtype=np.float64)

    gp = GaussianProcessRegressor(
        kernel=make_kernel(kernel_name, nu),
        n_restarts_optimizer=3,
        normalize_y=True,
        random_state=RANDOM_STATE,
    )
    gp.fit(X_train, y_train)
    pred = gp.predict(X_test)
    return pred, str(gp.kernel_)


cv_rows = []
for target_name, target_col in TARGETS.items():
    for spec in kernel_specs:
        fold_errors = []
        fitted_kernels = []
        for fold in range(N_FOLDS):
            train_df = df_cv[df_cv["spatial_fold"] != fold]
            test_df = df_cv[df_cv["spatial_fold"] == fold]
            pred, fitted_kernel = fit_predict_gp(
                train_df,
                test_df,
                target_col,
                spec["kernel"],
                None if pd.isna(spec["nu"]) else float(spec["nu"]),
            )
            y_true = test_df[target_col].to_numpy(dtype=np.float64)
            fold_errors.append(pd.DataFrame({
                "fold": fold,
                "y_true": y_true,
                "y_pred": pred,
                "error": pred - y_true,
            }))
            fitted_kernels.append(fitted_kernel)

        err_df = pd.concat(fold_errors, ignore_index=True)
        rmse = mean_squared_error(err_df["y_true"], err_df["y_pred"], squared=False)
        mae = mean_absolute_error(err_df["y_true"], err_df["y_pred"])
        bias = float(np.mean(err_df["error"]))
        cv_rows.append({
            "target": target_name,
            "kernel": spec["kernel"],
            "nu": spec["nu"],
            "RMSE": float(rmse),
            "MAE": float(mae),
            "Bias": bias,
            "n_folds": N_FOLDS,
            "max_train_per_fold": MAX_TRAIN,
            "example_fitted_kernel": fitted_kernels[0],
        })

cv_results = pd.DataFrame(cv_rows).sort_values(["target", "RMSE"]).reset_index(drop=True)
cv_results.to_csv(TABLE_DIR / "real_grid_gp_spatial_cv_kernel_selection.csv", index=False, encoding="utf-8-sig")
cv_results


TypeError: got an unexpected keyword argument 'squared'

In [ ]:
best_cv = cv_results.loc[cv_results.groupby("target")["RMSE"].idxmin()].sort_values("target")
best_cv[["target", "kernel", "nu", "RMSE", "MAE", "Bias"]]


## 8. 合併 AIC 與 Spatial CV 結果

最後把 AIC 選到的 kernel 與 spatial CV 選到的 kernel 放在一起比較。

判斷方式：

- 如果 AIC 與 spatial CV 都選同一個 kernel，代表證據比較一致。
- 如果 AIC 與 spatial CV 不一致，代表該參數的空間結構可能較不穩，應回頭看 variogram 與 residual map。


In [ ]:
summary = best_aic[["target", "kernel", "nu", "AIC"]].rename(
    columns={"kernel": "AIC_kernel", "nu": "AIC_nu"}
).merge(
    best_cv[["target", "kernel", "nu", "RMSE"]].rename(
        columns={"kernel": "CV_kernel", "nu": "CV_nu", "RMSE": "CV_RMSE"}
    ),
    on="target",
    how="inner",
)
summary["same_choice"] = (
    (summary["AIC_kernel"] == summary["CV_kernel"])
    & (summary["AIC_nu"].fillna(-999) == summary["CV_nu"].fillna(-999))
)
summary.to_csv(TABLE_DIR / "real_grid_gp_aic_cv_summary.csv", index=False, encoding="utf-8-sig")
summary


## 9. 報告時可以怎麼說

這個 workflow 的重點不是無腦 grid search，而是把 kernel 選擇拆成三個層次：

```text
Variogram gives intuition.
AIC gives quantitative model preference.
Spatial CV gives predictive evidence.
```

中文可以說：

> Variogram 用來看空間結構與平滑程度；AIC 用來量化真實資料比較支持哪一種 kernel；spatial CV 則用來驗證該 kernel 在空間外推上的預測能力。因此 RBF 或 Matérn 的選擇不是單純暴力搜尋，而是結合空間診斷、統計模型選擇與預測驗證。
